# Teste 1 — Estimativa do valor de empréstimo

Análise supervisionada de regressão com a base fornecida. Execute as células em ordem; mantenha `base_credito.csv` na mesma pasta deste notebook.

## 1. Carregar e conhecer os dados

Entradas: `idade` (anos), `renda_mensal` (renda mensal), `score_credito` (pontuação de crédito) e `historico_pagamentos` (porcentagem de pagamentos feitos em dia, de 0 a 100%). Alvo: `valor_emprestimo` (valor aprovado). Essa definição da coluna `historico_pagamentos` foi informada pelo professor em complemento ao PDF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

df = pd.read_csv('base_credito.csv')
display(df)
print('Dimensão:', df.shape)
print('Tipos e valores ausentes:')
display(pd.DataFrame({'tipo':df.dtypes.astype(str), 'ausentes':df.isna().sum()}))
display(df.describe())

Os dados contêm **50,000 registros** e cinco colunas numéricas: `idade` e `score_credito` são inteiros; as outras três são números decimais. Há **0 valores ausentes**.

## 2. Análise inicial

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df['renda_mensal'], df['valor_emprestimo'], alpha=.16, s=8)
ax.set(xlabel='Renda mensal', ylabel='Valor do empréstimo', title='Renda mensal × empréstimo aprovado')
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
print('Correlação de Pearson:', round(df['renda_mensal'].corr(df['valor_emprestimo']),3))

A relação entre renda e empréstimo é positiva, mas há dispersão: a renda isoladamente não determina o valor. Correlação de Pearson nesta base: **0.887**.

## 3. Preparar dados e treinar

Divisão aleatória reproduzível de **80% treino** e **20% teste**. O conjunto de teste só é usado na avaliação. A floresta aleatória captura relações não lineares sem exigir padronização das escalas.

In [ ]:
entradas = ['idade','renda_mensal','score_credito','historico_pagamentos']
X, y = df[entradas], df['valor_emprestimo']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
modelo = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)
previsoes = modelo.predict(X_test)
print('Treino:',len(X_train),'Teste:',len(X_test))

## 4. Avaliar e comparar

In [ ]:
mae = mean_absolute_error(y_test, previsoes)
rmse = np.sqrt(mean_squared_error(y_test, previsoes))
r2 = r2_score(y_test, previsoes)
print(f'MAE: {mae:,.2f} | RMSE: {rmse:,.2f} | R²: {r2:.4f}')
comparacao = pd.DataFrame({'Real':y_test.to_numpy(), 'Previsto':previsoes})
comparacao['Erro absoluto'] = (comparacao.Real - comparacao.Previsto).abs()
display(comparacao.head(10).round(2))
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y_test, previsoes, alpha=.15, s=8)
limites = [min(y_test.min(),previsoes.min()), max(y_test.max(),previsoes.max())]
ax.plot(limites, limites, color='crimson', label='Previsão perfeita')
ax.set(xlabel='Valor real', ylabel='Valor previsto', title='Teste: valores reais × previstos')
ax.legend()
plt.tight_layout()
plt.show()

O **MAE de 3,015.89** significa erro absoluto médio nessa unidade monetária. O **RMSE de 4,043.67** penaliza mais os erros grandes. O **R² de 0.9212** indica que aproximadamente 92.1% da variação no conjunto de teste é explicada pelo modelo. Nos dez primeiros exemplos, os erros absolutos variam de 341.11 a 6,512.15; há aproximação, mas nenhuma previsão é exata.

## 5. Novo cliente

In [ ]:
novo_cliente = pd.DataFrame([{'idade':35, 'renda_mensal':8000, 'score_credito':720, 'historico_pagamentos':85}])
display(novo_cliente)
valor_estimado = modelo.predict(novo_cliente)[0]
print(f'Valor estimado: {valor_estimado:,.2f}')

Para uma pessoa de 35 anos, renda mensal de 8.000, score de 720 e 85% dos pagamentos em dia, o modelo estima **38,521.89**. É uma estimativa estatística com base nos exemplos fornecidos, não uma decisão automática de concessão.

## 6. Conclusão

O desempenho no teste é adequado para uma estimativa exploratória nesta base, considerando o erro médio e o R² apresentados. As limitações incluem ausência de informações sobre origem, período e unidade monetária dos dados, variáveis que podem faltar para avaliar risco, possíveis vieses históricos e a incapacidade de garantir desempenho em clientes diferentes dos observados. Antes de usar em concessão real, seria preciso validação temporal, análise de grupos, regras de risco e supervisão humana.